In [10]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
import plotly.io as pio
import pickle 
import os
from pathlib import Path
from scipy import stats
import numpy as np
import datetime
import numpy as np
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pandas.tseries.holiday import USFederalHolidayCalendar
from typing import Literal
import importlib

**This notebook is primarily focusssed on modelling the conditional mean and variance using ARMA - GJR - GARCH model** <br>

ARMA models the conditonal mean <br>
GJR - GARCH models the asymmetric vols on +/-same shock

### Why are we modelling conditional mean and variance?

 Raw mean and variance assumes iid across time, but financial data violates it.
 1. Has volatility clustering
 2. Autocorrelation to recent past
 3. Asymmetric effect (Why GJR GARCH is used)

And also, if you pass in raw mean and vairance into copulas- it breaks. Why?
1. Copulas are modelling changing dependence- which needs conditioned variance and mean
2. **VIOLATES SKLARs THEOREM: SKLARS theorem assumes marginals transaform, F1, F2, etc. iid random varianble into uniform variables, Ui ~ Uniform(0,1). If vols are clustered, observations within same period are serially dependent, violating iid assumption and also distorts the PIT,prob integral transform** 


## Read Data

In [2]:
current_path = Path.cwd()
try:
    project_root = next(p for p in current_path.parents if p.name == "CommodityProject")
except StopIteration:
    print(Path.cwd())
except e as exception:
    print(e)

In [4]:
%cd $project_root

/Users/anandjoy/Documents/projects/Ian/CommodityProject


In [6]:
#'rb' (write binary) mode and dump the dict into it
with open("SystemicRiskCopulaProject/Data/dfs.pkl", "rb") as f:
    dfs = pickle.load(f)

In [7]:
import SystemicRiskCopulaProject.Scripts.BasicOperations as basic
import pandas as pd

In [9]:
dfs.keys()

dict_keys(['NG', 'Copper', 'Brent', 'Corn', 'WTI', 'Cattle', 'Cocoa'])

## ARMA modelling

$$R_{i,t} = \alpha_i + \sum_{j=1}^{p} \beta_{i,j} R_{i,t-j} + \sum_{k=1}^{q} \zeta_{i,k} \epsilon_{i,t-k} + \epsilon_{i,t}$$

$p$: The lag order of the autoregressive process (how many past time periods are included).<br>$\beta_{i,j}$: The coefficient measuring the persistence/impact of past returns $j$ periods ago on the current return.<br>$R_{i,t-j}$: The historical return of the asset at lag $t-j$. <br>
$q$: The lag order of the moving average process.<br>$\zeta_{i,k}$: The coefficient measuring the sensitivity to unexpected market shocks $k$ periods ago.<br>$\epsilon_{i,t-k}$: The past unpredicted residual or unexpected shock (innovation) from $k$ periods ago.<br>$\epsilon_{i,t}$ residuals gets passed onto GJR-GARCH assuming non-constant vol under normal distribution,ie, heteroskedasity

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

def select_best_arma(series, max_p=3, max_q=3):
    """
    Fits ARMA(p, q) models across candidate orders and returns the 
    best model according to BIC, following the paper's specification.
    """
    best_bic = np.inf
    best_order = (0, 0)
    best_model_fit = None

    for p in range(max_p + 1):
        for q in range(max_q + 1):
            try:
                # In statsmodels, ARIMA(p, 0, q) is an ARMA(p, q)
                model = ARIMA(series, order=(p, 0, q), trend='c')
                fitted = model.fit()
                
                if fitted.bic < best_bic:
                    best_bic = fitted.bic
                    best_order = (p, q)
                    best_model_fit = fitted
            except Exception:
                continue

    print(f"Optimal Order selected: ARMA{best_order} with BIC = {best_bic:.2f}")
    return best_order, best_model_fit